<a href="https://colab.research.google.com/github/awa-easymoneysniper/OSU-Mania-Auto-Beatmap-Generator/blob/main/mainia_audio_feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip -q install transformers torch torchaudio librosa

In [ ]:
"""
Step 1 (revised): extract MERT embeddings for every mp3 in AUDIO_DIR.
Replaces the OpenL3 approach -- OpenL3 relies on an unmaintained,
compiled dependency chain (resampy/kapre/old TensorFlow) that fails to
build under current Colab Python versions. MERT is a pure PyTorch /
HuggingFace model with no such build step.

Run in Colab after:
    !pip install transformers torch torchaudio librosa tqdm
"""

import os
import numpy as np
import torch
import librosa
from transformers import AutoModel, Wav2Vec2FeatureExtractor
from tqdm import tqdm

SAVE_ROOT = "/content/drive/MyDrive/osu_mania_dataset"
AUDIO_DIR = os.path.join(SAVE_ROOT, "audio")
EMBED_DIR = os.path.join(SAVE_ROOT, "audio_embeddings")

MODEL_NAME = "m-a-p/MERT-v1-95M"   # swap to MERT-v1-330M for a larger model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

_model = None
_processor = None
_target_sr = None


def load_mert():
    """
    Load the MERT model and its matching feature extractor once, and cache
    them in module-level globals so process_dataset() doesn't reload the
    model per file.
    """
    global _model, _processor, _target_sr
    if _model is None:
        _model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True).to(DEVICE)
        _model.eval()
        _processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME, trust_remote_code=True)
        _target_sr = _processor.sampling_rate  # 24000 for MERT-v1
    return _model, _processor, _target_sr


def load_mp3(path, target_sr):
    """
    Decode an mp3 to a mono float32 array at target_sr using librosa
    (ffmpeg-backed), which handles mp3 directly.
    """
    audio, sr = librosa.load(path, sr=target_sr, mono=True)
    return audio.astype(np.float32)


def extract_embedding(mp3_path):
    """
    Compute MERT hidden-state embeddings for one mp3 file.

    Returns:
        layerwise: (num_layers, T, hidden_dim) array -- one embedding
            sequence per transformer layer (13 layers x 768 dim for 95M).
        pooled: (num_layers, hidden_dim) array -- each layer mean-pooled
            over time, a cheaper representation to start experiments with.
    """
    model, processor, target_sr = load_mert()
    audio = load_mp3(mp3_path, target_sr)

    inputs = processor(audio, sampling_rate=target_sr, return_tensors="pt")
    input_values = inputs["input_values"].to(DEVICE)

    with torch.no_grad():
        outputs = model(input_values, output_hidden_states=True)

    # hidden_states: tuple of (1, T, hidden_dim), one per layer
    layerwise = torch.stack(outputs.hidden_states).squeeze(1)  # (num_layers, T, hidden_dim)
    pooled = layerwise.mean(dim=1)                              # (num_layers, hidden_dim)

    return layerwise.cpu().numpy(), pooled.cpu().numpy()


def process_dataset(audio_dir=AUDIO_DIR, embed_dir=EMBED_DIR, overwrite=False):
    """
    Walk audio_dir, extract MERT embeddings for every mp3, save each as an
    .npz (layerwise + pooled) in embed_dir under the same base filename.

    Resumable: files whose output .npz already exists are skipped unless
    overwrite=True. Files that fail to decode or process are logged and
    skipped rather than crashing the whole batch.
    """
    os.makedirs(embed_dir, exist_ok=True)
    load_mert()  # load once up front, not inside the loop

    mp3_files = [f for f in os.listdir(audio_dir) if f.lower().endswith(".mp3")]
    failed = []

    for fname in tqdm(mp3_files, desc="Extracting MERT embeddings"):
        base = os.path.splitext(fname)[0]
        out_path = os.path.join(embed_dir, base + ".npz")

        if os.path.exists(out_path) and not overwrite:
            continue

        mp3_path = os.path.join(audio_dir, fname)
        try:
            layerwise, pooled = extract_embedding(mp3_path)
            np.savez(out_path, layerwise=layerwise, pooled=pooled)
        except Exception as e:
            failed.append((fname, str(e)))

    if failed:
        print(f"\n{len(failed)} files failed:")
        for fname, err in failed:
            print(f"  {fname}: {err}")

    return failed


if __name__ == "__main__":
    process_dataset()

config.json:   0%|          | 0.00/1.82k [00:00<?, ?B/s]

configuration_MERT.py:   0%|          | 0.00/5.34k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-95M:
- configuration_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_MERT.py:   0%|          | 0.00/18.0k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-95M:
- modeling_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

Extracting MERT embeddings:   0%|          | 33/8799 [07:07<31:31:32, 12.95s/it]


KeyboardInterrupt: 

In [ ]:
"""
Step 1 (revised again): chunked, batched MERT embedding extraction.

Two changes from the previous version:
  1. Each song is split into fixed-length chunks and run through MERT as a
     batch, instead of one whole-song forward pass. Whole-song passes are
     slow because attention cost grows with sequence length, and MERT was
     pretrained on short windows anyway.
  2. Saves are atomic (write to a temp file, then rename) so a crash or
     disconnect mid-save can't leave a corrupted .npz behind.

Run in Colab after:
    !pip install transformers torch torchaudio librosa tqdm
"""

import os
import math
import numpy as np
import torch
import librosa
from transformers import AutoModel, Wav2Vec2FeatureExtractor
from tqdm import tqdm

SAVE_ROOT = "/content/drive/MyDrive/osu_mania_dataset"
AUDIO_DIR = os.path.join(SAVE_ROOT, "audio")
EMBED_DIR = os.path.join(SAVE_ROOT, "audio_embeddings")

MODEL_NAME = "m-a-p/MERT-v1-95M"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHUNK_SECONDS = 10      # length of each window fed to the model
CHUNK_BATCH_SIZE = 8    # how many chunks go through the model at once

_model = None
_processor = None
_target_sr = None


def load_mert():
    """
    Load the MERT model and feature extractor once, cached in module-level
    globals. Prints the device in use so a CPU fallback is obvious.
    """
    global _model, _processor, _target_sr
    if _model is None:
        print(f"Loading MERT on device: {DEVICE}")
        if DEVICE == "cpu":
            print("WARNING: no GPU detected. This will be very slow for 9000 songs. "
                  "Check Runtime > Change runtime type > GPU in Colab.")
        _model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True).to(DEVICE)
        _model.eval()
        _processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME, trust_remote_code=True)
        _target_sr = _processor.sampling_rate  # 24000 for MERT-v1
    return _model, _processor, _target_sr


def load_mp3(path, target_sr):
    """Decode an mp3 to a mono float32 array at target_sr using librosa."""
    audio, sr = librosa.load(path, sr=target_sr, mono=True)
    return audio.astype(np.float32)


def chunk_audio(audio, sr, chunk_seconds=CHUNK_SECONDS):
    """
    Split a raw audio array into fixed-length, non-overlapping chunks.
    Zero-pads the final chunk so all chunks are the same length, which is
    required to batch them through the model together.

    Returns:
        chunks: (n_chunks, chunk_len) array
        valid_len: length of audio before padding, in samples (needed to
            trim padding-derived frames from the pooled output later)
    """
    chunk_len = int(chunk_seconds * sr)
    n_chunks = max(1, math.ceil(len(audio) / chunk_len))
    padded_len = n_chunks * chunk_len
    valid_len = len(audio)
    if padded_len > len(audio):
        audio = np.pad(audio, (0, padded_len - len(audio)))
    chunks = audio.reshape(n_chunks, chunk_len)
    return chunks, valid_len


def extract_embedding(mp3_path):
    """
    Compute MERT embeddings for one mp3 file by chunking it and running
    chunks through the model in batches.

    Returns:
        pooled_per_chunk: (num_layers, n_chunks, hidden_dim) -- each chunk
            mean-pooled over time, per layer. Concatenating/averaging
            across the chunk axis gives a whole-song representation.
    """
    model, processor, target_sr = load_mert()
    audio = load_mp3(mp3_path, target_sr)
    chunks, _ = chunk_audio(audio, target_sr)

    pooled_chunks = []
    for start in range(0, len(chunks), CHUNK_BATCH_SIZE):
        batch = chunks[start:start + CHUNK_BATCH_SIZE]
        inputs = processor(
            list(batch), sampling_rate=target_sr,
            return_tensors="pt", padding=True,
        )
        input_values = inputs["input_values"].to(DEVICE)

        with torch.no_grad():
            outputs = model(input_values, output_hidden_states=True)

        # hidden_states: tuple of (batch, T, hidden_dim), one per layer
        layerwise = torch.stack(outputs.hidden_states)   # (num_layers, batch, T, hidden_dim)
        pooled = layerwise.mean(dim=2)                    # (num_layers, batch, hidden_dim)
        pooled_chunks.append(pooled.cpu().numpy())

    # concat along the chunk axis -> (num_layers, n_chunks, hidden_dim)
    return np.concatenate(pooled_chunks, axis=1)


def atomic_save(out_path, **arrays):
    """
    Write an .npz to a temp file and rename it into place. Rename is
    atomic on POSIX filesystems, so a crash mid-write can never leave a
    half-written .npz at out_path -- either the old file, or none, or the
    fully-written new one.
    """
    # tmp_path must already end in .npz -- np.savez silently appends .npz
    # to any filename that doesn't have it, which would otherwise produce
    # e.g. "000095.npz.tmp.npz" and break the rename below
    tmp_path = out_path.replace(".npz", ".tmp.npz")
    np.savez(tmp_path, **arrays)
    os.replace(tmp_path, out_path)


def process_dataset(audio_dir=AUDIO_DIR, embed_dir=EMBED_DIR, overwrite=False):
    """
    Walk audio_dir, extract MERT embeddings for every mp3, save each as an
    .npz in embed_dir under the same base filename.

    Resumable: files whose output .npz already exists are skipped unless
    overwrite=True, so interrupting and rerunning this cell (e.g. after a
    Colab disconnect) continues from where it left off.
    """
    os.makedirs(embed_dir, exist_ok=True)
    load_mert()

    mp3_files = sorted(f for f in os.listdir(audio_dir) if f.lower().endswith(".mp3"))
    failed = []

    for fname in tqdm(mp3_files, desc="Extracting MERT embeddings"):
        base = os.path.splitext(fname)[0]
        out_path = os.path.join(embed_dir, base + ".npz")

        if os.path.exists(out_path) and not overwrite:
            continue

        mp3_path = os.path.join(audio_dir, fname)
        try:
            pooled_per_chunk = extract_embedding(mp3_path)
            atomic_save(out_path, pooled_per_chunk=pooled_per_chunk)
        except Exception as e:
            failed.append((fname, str(e)))

    if failed:
        print(f"\n{len(failed)} files failed:")
        for fname, err in failed:
            print(f"  {fname}: {err}")

    return failed


if __name__ == "__main__":
    process_dataset()

Loading MERT on device: cuda


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Extracting MERT embeddings:   1%|          | 100/8799 [03:48<4:45:01,  1.97s/it]/tmp/ipykernel_823/3628981566.py:58: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(path, sr=target_sr, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Extracting MERT embeddings:   3%|▎         | 247/8799 [12:32<8:28:59,  3.57s/it]/tmp/ipykernel_823/3628981566.py:58: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(path, sr=target_sr, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, d